In [0]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()

print(f"env_code: {lz_key}")  # This won't be redacted
print(f"env_name: {env_name}")  # This won't be redacted

KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"KeyVault_name: {KeyVault_name}") 


# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")

# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"

# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# read_hive = False

# Setting variables for use in subsequent cells
raw_mnt = f"abfss://raw@ingest{lz_key}raw{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
landing_mnt = f"abfss://landing@ingest{lz_key}landing{env_name}.dfs.core.windows.net/SQLServer/Sales/IRIS/dbo/"
bronze_mnt = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
silver_mnt = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
gold_mnt = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
gold_outputs = "ARIADM/ARM/JOH"
hive_schema = "ariadm_arm_joh"
# key_vault = "ingest00-keyvault-sbox"

html_mnt = f"abfss://html-template@ingest{lz_key}landing{env_name}.dfs.core.windows.net/"

# Print all variables
variables = {
    # "read_hive": read_hive,
    "raw_mnt": raw_mnt,
    "landing_mnt": landing_mnt,
    "bronze_mnt": bronze_mnt,
    "silver_mnt": silver_mnt,
    "gold_mnt": gold_mnt,
    "html_mnt": html_mnt,
    "gold_outputs": gold_outputs,
    "hive_schema": hive_schema,
    "key_vault": KeyVault_name
}

display(variables)

try:
    env_value = dbutils.secrets.get(KeyVault_name, "Environment")
    env = "dev" if env_value == "development" else None
    print(f"Environment: {env}")
except:
    env = "unkown"

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_joh.stg_create_joh_html_content t1
JOIN aria_stg01.ariadm_arm_joh.stg_create_joh_html_content t2

  ON t1.AdjudicatorId = t2.AdjudicatorId

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_td.stg_create_td_iris_html_content t1
JOIN aria_stg01.ariadm_arm_td.stg_create_td_iris_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.aria_bails.create_bails_html_content t1
JOIN aria_stg01.aria_bails.create_bails_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.aria_s_bails.create_sbail_html_content t1
JOIN aria_stg01.aria_s_bails.create_sbail_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_fpa.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_fpa.stg_apl_create_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_uta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_uta.stg_apl_create_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_fta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_fta.stg_apl_create_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:

%sql
SELECT
  t1.CaseNo,

  length(
    regexp_replace(
      regexp_replace(t1.HTML_Content, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS meta_normalised_length,

  length(
    regexp_replace(
      regexp_replace(t2.HTML_Content, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS uc_normalised_length,

  regexp_replace(
    regexp_replace(t1.HTML_Content, '\\s+', ''),
    '&nbsp;',
    ''
  )
  =
  regexp_replace(
    regexp_replace(t2.HTML_Content, '\\s+', ''),
    '&nbsp;',
    ''
  ) AS normalised_match,
  t1.HTML_Content as metastore_html,
  t2.HTML_Content as uc_html

FROM hive_metastore.ariadm_arm_fta.stg_apl_create_html_content t1

JOIN aria_stg01.ariadm_arm_fta.stg_apl_create_html_content t2
  ON t1.CaseNo = t2.CaseNo

WHERE t1.CaseNo IN (
-- "EA/03627/2023",
-- "HU/04962/2016",
-- "LE/04007/2024",
-- "PA/01443/2020",
-- "PA/11516/2016",
-- "IA/09536/2022",
"LP/06709/2024")



In [0]:
%sql

with t as (SELECT

  t1.CaseNo,
  length(regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')) AS meta_normalised_length,
  length(regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')) AS uc_normalised_length,

  case when length(regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', ''))
            =     
            length(regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', ''))
  then 1 else 0 end as normalised_length_match,

  case when regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
            =
            regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')  
  then 1 else 0 end as normalised_match

  -- t1.HTML_Content AS metastore_html,
  -- t2.HTML_Content AS uc_html

FROM hive_metastore.ariadm_arm_fta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_fta.stg_apl_create_html_content t2
ON t1.CaseNo = t2.CaseNo)

select t.* from t where t.normalised_length_match = 0

